# Teacher data, stage `explain`

Teacher Qwen/Qwen3-14B-GGUF:Qwen3-14B-Q4_K_M.gguf on llamacpp (tp=2). Needs GPU T4 x2, Internet, secret `hg` (HF write token). Output goes to a private HF dataset repo. On T4s the engine is llama.cpp built with CUDA for sm_75: vLLM 0.33 does not start on these GPUs (probe v1/v2).

In [ ]:
%pip install -q "distill @ git+https://github.com/MSR327/explain-distill.git@fa71d4c"
COMMIT, STAGE, MODEL, TP, ENGINE = "fa71d4c", "explain", "Qwen/Qwen3-14B-GGUF:Qwen3-14B-Q4_K_M.gguf", 2, "llamacpp"

In [ ]:
# Fail fast: HF token + write access before loading a 14B model.
import os

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("hg")
api = HfApi()
REPO = f"{api.whoami()['name']}/explain-distill-data"
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
print("HF auth ok ->", REPO)

In [ ]:
# llama.cpp CUDA for compute capability 7.5: cached binaries if present, else build (~25 min) + cache.
import os, re, subprocess, tarfile
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import EntryNotFoundError

def _build_sh(cmd, log):
    import time
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
        t0 = time.time()
        while p.poll() is None:  # progress every minute: a silent 30-min build looks the same as a hung one
            time.sleep(60)
            pct = re.findall(r"\[ *(\d+)%\]", open(log).read()[-4000:])
            print(f"  {log}: {(time.time() - t0) / 60:.0f} min" + (f", {pct[-1]}%" if pct else ""), flush=True)
    if p.returncode:
        print(open(log).read()[-6000:])
        raise RuntimeError(f"failed: {cmd}")

_build_sh("(nvcc --version || /usr/local/cuda/bin/nvcc --version) 2>&1 || true", "nvcc.log")
cuda = (re.findall(r"release ([0-9.]+)", open("nvcc.log").read()) or ["unknown"])[0]
TARGETS = "llama-server".split()
_api = HfApi()
_repo = f"{_api.whoami()['name']}/explain-distill-data"
_tar = f"llama-cpp-bin/b54752df40-cuda{cuda}-sm75.tar.gz"
print("CUDA", cuda, "cache", _tar)
if not os.path.exists("llama.cpp/.git"):
    _build_sh("git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp "
       "&& git fetch -q --depth 1 origin b54752df40e4d8e53d34ef5a598336b4e4871d39 && git checkout -q FETCH_HEAD", "clone.log")
try:
    with tarfile.open(hf_hub_download(_repo, _tar, repo_type="dataset")) as t:
        t.extractall("llama.cpp", filter="data")
    print("using cached llama.cpp binaries")
except EntryNotFoundError:
    _build_sh("cd llama.cpp && cmake -B build -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
       "-DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release", "cmake.log")
    _build_sh("cd llama.cpp && cmake --build build -j 4 --target llama-server llama-quantize", "build.log")
    with tarfile.open("llama-bin.tar.gz", "w:gz") as t:
        t.add("llama.cpp/build/bin", arcname="build/bin")
    _api.upload_file(path_or_fileobj="llama-bin.tar.gz", path_in_repo=_tar, repo_id=_repo, repo_type="dataset",
                     commit_message=f"llama.cpp b54752df40 CUDA {cuda} sm_75 binaries")
    print("built and cached llama.cpp binaries")
for t in TARGETS:
    assert os.access(f"llama.cpp/build/bin/{t}", os.X_OK), t
print("llama.cpp ready:", *TARGETS)

In [ ]:
from huggingface_hub import hf_hub_download
import shutil
os.makedirs("data", exist_ok=True)
for name in ['pairs.jsonl']:
    shutil.copy(hf_hub_download(REPO, name, repo_type="dataset"), f"data/{name}")
    print("downloaded", name)

In [ ]:
# Smoke pass on a few items: catches a broken stack in minutes, not hours.
import subprocess, sys
def run(extra, data):
    for d in ("data", data):
        os.makedirs(d, exist_ok=True)
    for name in ['pairs.jsonl'] if data != "data" else []:  # the full run reads data/ in place
        shutil.copy(f"data/{name}", f"{data}/{name}")
    cmd = (f"{sys.executable} -m distill.teacher {STAGE} --model {MODEL} --tp {TP} --engine {ENGINE} "
           f"--data {data} {extra}")
    # Stream progress ("generated N/M X min") live so a long run is visibly alive; stderr goes to a file.
    lines = []
    with open(f"{data}/stderr.log", "w") as err:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=err, text=True, bufsize=1)
        for x in p.stdout:
            if x.startswith(("TEACHER", "generated")):
                print(x.rstrip(), flush=True)
            if x.startswith("TEACHER"):
                lines.append(x.rstrip())
        p.wait()
    if p.returncode or not lines:
        print(open(f"{data}/stderr.log").read()[-4000:])
        import glob
        for log in sorted(glob.glob("llama-server-*.log")):
            print(f"--- {log} (tail) ---", open(log).read()[-2500:], sep="\n")
        raise RuntimeError(f"stage failed: {cmd}")
    return lines

run("--limit 8", "smoke")
print(open("smoke/teacher.jsonl").read()[:1500])

In [ ]:
import time
t = time.time()
run("", "data")
print(f"full run {(time.time() - t) / 60:.1f} min")

In [ ]:
api.upload_file(path_or_fileobj="data/teacher.jsonl", path_in_repo="teacher.jsonl", repo_id=REPO,
                repo_type="dataset", commit_message=f"explain ({MODEL}, code commit {COMMIT})")
print("uploaded teacher.jsonl")